# EXAONE-3.5-2.4B 공문/보고서 LoRA 파인튜닝

**런타임 → GPU(T4)** 로 바꾸고 위에서부터 셀 순서대로 실행.

## 흐름
1. 패키지 설치
2. HF 로그인 (Write 토큰)
3. train_data.json 업로드
4. 모델/토크나이저 로드 (EXAONE-3.5-2.4B, fp16)
5. LoRA 설정 + 학습
6. 학습 전/후 생성 비교
7. LoRA 어댑터를 HF에 push

## 1. 패키지 설치

In [ ]:
# EXAONE 커스텀 코드가 신규 transformers API를 써서, 최신 버전 필수
# 실행 후 반드시 커널 재시작:
#   Kaggle: Run → Restart Kernel  (또는 Ctrl+M, R)
#   Colab:  Runtime → Restart session
!pip install -q -U transformers peft accelerate bitsandbytes datasets trl huggingface_hub

## 2. HF 로그인

In [ ]:
# Colab: notebook_login 위젯에 토큰 붙여넣기
# Kaggle: 미리 Secrets에 HF_TOKEN 등록해 두면 자동으로 사용
import os
from huggingface_hub import login

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
    print('Kaggle Secret HF_TOKEN 사용')
except Exception:
    pass

if token:
    login(token=token, add_to_git_credential=False)
else:
    from huggingface_hub import notebook_login
    notebook_login()
print('HF 로그인 완료')

## 3. train_data.json 업로드

좌측 파일 패널(폴더 아이콘) → 업로드 버튼 → 로컬의 `training/train_data.json` 선택

In [ ]:
import json, glob
# Kaggle Add Input(/kaggle/input/...) 또는 직접 업로드(작업 디렉토리) 둘 다 자동 탐지
candidates = (
    glob.glob('/kaggle/input/**/train_data.json', recursive=True)
    + glob.glob('./train_data.json')
    + glob.glob('/content/train_data.json')
)
assert candidates, 'train_data.json을 찾을 수 없습니다. Kaggle: Add Input으로 데이터셋 attach, Colab: 좌측에서 업로드'
path = candidates[0]
print(f'데이터 경로: {path}')
with open(path, 'r', encoding='utf-8') as f:
    raw = json.load(f)
print(f'데이터: {len(raw)}개')
for i, x in enumerate(raw):
    print(f'  [{i}] doc_type={x["doc_type"]}, instruction={x["instruction"][:50]}...')

## 4. 모델 + 토크나이저 로드

In [ ]:
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

print('transformers version:', transformers.__version__)
# 최신 transformers는 EXAONE을 내장 지원 → trust_remote_code 끄고 사용 (LGAI 커스텀 코드의 RopeParameters 충돌 회피)

MODEL_ID = 'LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=False,  # ← 핵심: 깨진 커스텀 코드 우회
)
print('model:', model.dtype, 'device:', next(model.parameters()).device)
print('arch:', type(model).__name__)

## 5. 시스템 프롬프트 + 데이터셋 포맷

양식별 시스템 프롬프트를 prepend 해서 모델이 doc_type 신호를 학습하도록 함.

In [ ]:
SYS_LETTER = '''당신은 한국 공문 작성 전문가입니다. 수신/제목/본문 도입(한 단락)/세부 항목(1./2./3.)/※ 안내/붙임 ... 끝. 양식으로 작성하세요. 발신명의는 생략. 마크다운 금지.'''
SYS_REPORT = '''당신은 한국 기업/공공 보고서 작성 전문가입니다. 작성일자/부서/1. 추진 배경 / 2. 개요·결과 / 3. 조치 계획 / 4. 향후 일정 식으로 가./나./다. 세부와 - 글머리 사용. 마크다운 금지.'''

from torch.utils.data import Dataset

class GongmunDataset(Dataset):
    def __init__(self, data, tok, max_len=1024):
        self.samples = []
        for item in data:
            sys = SYS_LETTER if item['doc_type'] == '공문' else SYS_REPORT
            messages = [
                {'role': 'system', 'content': sys},
                {'role': 'user', 'content': item['instruction']},
                {'role': 'assistant', 'content': item['output']},
            ]
            text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
            enc = tok(text, truncation=True, max_length=max_len, padding='max_length', return_tensors='pt')
            input_ids = enc['input_ids'].squeeze()
            attn = enc['attention_mask'].squeeze()
            labels = input_ids.clone()
            labels[labels == tok.pad_token_id] = -100
            self.samples.append({'input_ids': input_ids, 'attention_mask': attn, 'labels': labels})
    def __len__(self): return len(self.samples)
    def __getitem__(self, i): return self.samples[i]

dataset = GongmunDataset(raw, tokenizer)
print(f'학습 샘플: {len(dataset)}개')

## 6. LoRA 설정 + 학습

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import TrainingArguments, Trainer

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'out_proj'],  # EXAONE attention
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

args = TrainingArguments(
    output_dir='./out',
    num_train_epochs=10,   # 데이터 적으니 많이 반복
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=1,
    save_strategy='no',
    report_to='none',
    remove_unused_columns=False,
)
trainer = Trainer(model=model, args=args, train_dataset=dataset)
trainer.train()

## 7. 학습 전/후 비교 (간단 추론 테스트)

In [ ]:
def gen(instruction, doc_type='공문', max_new=600):
    sys = SYS_LETTER if doc_type == '공문' else SYS_REPORT
    messages = [
        {'role': 'system', 'content': sys},
        {'role': 'user', 'content': instruction},
    ]
    ids = tokenizer.apply_chat_template(messages, return_tensors='pt', add_generation_prompt=True).to(model.device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new, do_sample=True, temperature=0.7, top_p=0.8)
    return tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

# 공문 테스트
print('=== 공문 ===')
print(gen('26.6.15(화) 10시 본사 강당에서 신입사원 환영회. 임원진 참석.', '공문'))
print()
print('=== 보고서 ===')
print(gen('26.6월 사내 AI 도구 도입 결과 보고. 도입부서 5개, 만족도 85%.', '보고서'))

## 8. LoRA 어댑터를 HF에 push

- 모델 전체 병합 후 ONNX 변환은 별도 작업 (시간/메모리 큼)
- 우선 LoRA만 올려서 Colab/PyTorch 환경에서 테스트 가능하게 함

In [ ]:
REPO = 'taeaiai/exaone-gongmun-lora'

# private repo 자동 생성 후 LoRA 저장
model.push_to_hub(REPO, private=True)
tokenizer.push_to_hub(REPO, private=True)
print(f'완료: https://huggingface.co/{REPO}')

## 9. (선택) LoRA 병합 + ONNX 변환

Space 무료 CPU에서 쓰려면 ONNX로 변환 필요. 메모리 부담 커서 별도 셀로 분리.

이 셀은 학습 결과가 만족스러울 때 실행 — 시간 30~60분.

In [ ]:
# LoRA 어댑터를 base에 병합
merged = model.merge_and_unload()
merged.save_pretrained('./merged_model')
tokenizer.save_pretrained('./merged_model')

# Optimum으로 ONNX 변환 (시간 오래 걸림)
!pip install -q optimum[onnxruntime]==1.23.0
!optimum-cli export onnx --model ./merged_model --task text-generation-with-past --opset 17 ./onnx_export

# q4 양자화
!optimum-cli onnxruntime quantize --onnx_model ./onnx_export --output ./onnx_q4 --avx512_vnni

# HF에 push (taeaiai/exaone-gongmun-ft-onnx)
from huggingface_hub import HfApi
api = HfApi()
api.create_repo('taeaiai/exaone-gongmun-ft-onnx', private=True, exist_ok=True)
api.upload_folder(folder_path='./onnx_q4', repo_id='taeaiai/exaone-gongmun-ft-onnx')
print('ONNX 변환 + 업로드 완료. Space의 MODEL_REPO 환경변수를 taeaiai/exaone-gongmun-ft-onnx 로 바꾸면 적용됨.')